# Investigating & Fixing Label Fragmentation


## Configuration and Label Inventory Analysis

Loads the current dataset and extracts all unique entity labels to understand the scope of label fragmentation before
consolidation.

This analysis is critical for understanding:

1. How many unique labels exist in the dataset
2. Which labels are semantically equivalent but differently named
3. The frequency distribution of each label (important for deciding consolidation)


In [4]:
from collections import Counter
from typing import Any

from datasets import load_dataset, DatasetDict

# =============================================================================
# Configuration
# =============================================================================

DATASET_ID: str = "Ari-S-123/better-english-pii-anonymizer"

# =============================================================================
# Load Dataset
# =============================================================================

print(f"Loading dataset from: {DATASET_ID}")
dataset: DatasetDict = load_dataset(DATASET_ID)

print(f"\nDataset structure:")
print(f"  Train split: {len(dataset['train']):,} examples")
print(f"  Test split: {len(dataset['test']):,} examples")

# =============================================================================
# Extract All Unique Labels
# =============================================================================


def extract_labels_from_split(split_data) -> Counter:
    """
    Extract all entity labels from a dataset split.

    Iterates through the privacy_mask field of each example and collects
    the 'label' value from each entity annotation.

    Args:
        split_data: A HuggingFace Dataset split containing 'privacy_mask' field.

    Returns:
        Counter object mapping label names to their occurrence counts.
    """
    label_counter: Counter = Counter()

    for example in split_data:
        privacy_mask: list[dict[str, Any]] = example.get("privacy_mask", [])
        for entity in privacy_mask:
            label: str = entity.get("label", "UNKNOWN")
            label_counter[label] += 1

    return label_counter


print("\nExtracting labels from train split...")
train_labels: Counter = extract_labels_from_split(dataset["train"])

print("Extracting labels from test split...")
test_labels: Counter = extract_labels_from_split(dataset["test"])

# Combine label counts from both splits
all_labels: Counter = train_labels + test_labels

# =============================================================================
# Display Label Statistics
# =============================================================================

print("\n" + "=" * 80)
print("LABEL INVENTORY ANALYSIS")
print("=" * 80)

print(f"\nTotal unique labels: {len(all_labels):,}")
print(f"Total entity occurrences: {sum(all_labels.values()):,}")

# Show labels sorted by frequency (descending)
print("\n--- Top 50 Most Frequent Labels ---")
for i, (label, count) in enumerate(all_labels.most_common(50), 1):
    pct: float = (count / sum(all_labels.values())) * 100
    print(f"  {i:3d}. {label:40s} {count:8,d} ({pct:5.2f}%)")

# Show labels with very low frequency (potential consolidation targets)
print("\n--- Labels with < 100 occurrences (consolidation candidates) ---")
low_freq_labels: list[tuple[str, int]] = [
    (label, count) for label, count in all_labels.items() if count < 100
]
low_freq_labels.sort(key=lambda x: x[1])

print(f"Count: {len(low_freq_labels)} labels with < 100 occurrences")
for label, count in low_freq_labels[:30]:  # Show first 30
    print(f"  {label:45s} {count:5,d}")

if len(low_freq_labels) > 30:
    print(f"  ... and {len(low_freq_labels) - 30} more")

# =============================================================================
# Identify Semantic Groups (Labels that likely mean the same thing)
# =============================================================================

print("\n--- Semantic Grouping Analysis ---")

# Define pattern-based groupings to identify similar labels
SEMANTIC_PATTERNS: dict[str, list[str]] = {
    "PHONE": ["PHONE", "PHONENUMBER", "PHONE_NUMBER", "PHONE_NUMBERS", "TELEPHONENUM",
              "UK_PHONE", "PHONE_EXT"],
    "NAME_GIVEN": ["FIRSTNAME", "GIVENNAME", "GIVEN_NAME", "FIRST_NAME",
                   "PERSON_FIRST_NAME", "PERSON_GIVEN_NAME"],
    "NAME_FAMILY": ["LASTNAME", "SURNAME", "LAST_NAME", "FAMILY_NAME",
                    "PERSON_LAST_NAME"],
    "NAME_FULL": ["NAME", "FULL_NAME", "PERSON_NAME", "PERSON_NAMES",
                  "PERSONAL_NAME", "PERSONAL_NAMES", "PERSON_FULLNAME",
                  "PERSON_FULL_NAME", "PERSON"],
    "SSN": ["SSN", "SOCIALNUM", "SOCIAL_SECURITY_NUMBER", "US_SSN",
            "SSN_LAST4", "SSN_LAST_FOUR", "US_SSN_LASTFOUR"],
    "PASSPORT": ["PASSPORT", "PASSPORTNUM", "PASSPORT_NUMBER", "PASSPORT_NUMBERS",
                 "PASSPORT_ID", "UK_PASSPORT", "GERMAN_PASSPORT"],
    "DRIVER_LICENSE": ["DRIVERLICENSENUM", "DRIVER_LICENSE", "DRIVERS_LICENSE",
                       "DRIVERS_LICENCE", "DRIVER_LICENCE", "DRIVERS_LICENSE_NUMBER",
                       "UK_DRIVING_LICENCE", "UK_DRIVING_LICENCE_DATE"],
    "CREDIT_CARD": ["CREDITCARDNUMBER", "CREDIT_CARD", "CARD_BRAND", "CARD_EXPIRY",
                    "CREDIT_CARD_CVV", "CREDIT_CARD_EXPIRY"],
    "EMAIL": ["EMAIL", "EMAIL_ADDRESS"],
    "ADDRESS": ["ADDRESS", "STREET_ADDRESS", "POSTAL_ADDRESS", "LOCATION_ADDRESS",
                "US_STREET_ADDRESS", "ADDRESS_RESIDENTIAL", "LOCATION"],
    "POSTAL_CODE": ["ZIPCODE", "POSTAL_CODE", "POSTAL_CODES", "POSTCODE", "POST_CODE",
                    "UK_POSTCODE", "US_ZIP_CODE", "PIN_CODE"],
    "BANK_ACCOUNT": ["BANK_ACCOUNT", "ACCOUNT_NUMBER", "BANK_ACCOUNT_NUMBER",
                     "BANK_ACCOUNT_NO", "BANK_ACCOUNT_IBAN", "ACCOUNTNUMBER", "ACCOUNT",
                     "ACCOUNT_ID"],
    "NATIONAL_ID": ["NATIONAL_ID", "NATIONAL_IDENTITY", "IDCARDNUM", "ID_CARD",
                    "ID_CARD_NUMBER", "ID_NUMBER", "ID_NUMBERS", "ID", "GOVT_ID",
                    "NATIONALID", "NATIONAL_IDENTITY_PAN", "NATIONAL_IDENTITY_SSN",
                    "NATIONAL_IDENTITY_SSN_AADHAR", "OTHER_NATIONAL_IDENTITY",
                    "NATIONAL_ID_PAN"],
    "TAX_ID": ["TAX_ID", "TAXNUM", "TAX_IDENTIFICATION", "TFN", "FISCAL_CODE",
               "IT_FISCAL_CODE"],
    "DATE_OF_BIRTH": ["DOB", "DATE_OF_BIRTH"],
    "IBAN": ["IBAN", "IBAN_CODE", "BANK_IBAN"],
    "CRYPTO_WALLET": ["CRYPTO_WALLET", "CRYPTO_WALLET_ADDRESS", "CRYPTO_WALLET_ETHEREUM",
                      "BITCOIN_ADDRESS", "BITCOIN_WALLET", "BITCOINADDRESS",
                      "ETHEREUM_WALLET", "LITECOINADDRESS"],
    "2FA_CODE": ["TFA_CODE", "2FA_CODE", "TWO_FACTOR_CODE", "TWO_FA_CODE",
                 "TXN_2FA_CODE", "OTP", "OTP_CODE", "TOTP", "TOTP_CODE",
                 "VERIFICATION_OTP", "AUTH_CODE", "AUTH_TOKEN", "INTERAC_CODE"],
    "VEHICLE_ID": ["VEHICLE_ID", "VEHICLE_IDENTIFICATION_NUMBER", "VEHICLE_ID_NUMBER",
                   "VEHICLE_VIN", "VEHICLE_REG", "VEHICLE_REGISTRATION", "VEHICLE_CHASSIS",
                   "VEHICLEVIN", "VEHICLEVRM"],
    "UPI_ID": ["UPI_ID", "UPI", "BANK_UPI_ID", "UPI_TRANSACTION_ID"],
    "USERNAME": ["USERNAME", "USERNAME_HANDLE", "MODERN_USERNAME", "DISCORD_HANDLE",
                 "DISCORD_TAG", "VENMO_HANDLE", "SOCIAL_MEDIA_HANDLE"],
    "INSURANCE": ["INSURANCE", "INSURANCE_NUMBER", "INSURANCE_POLICY",
                  "INSURANCE_POLICY_NUMBER", "NATIONAL_INSURANCE",
                  "NATIONAL_INSURANCE_NUMBER", "UK_NATIONAL_INSURANCE_NUMBER",
                  "UK_NI", "UK_NI_NUMBER", "UK_NHS_NUMBER", "MEDICARE_NUMBER",
                  "HEALTH_CARD_NUMBER"],
    "AADHAAR": ["AADHAAR", "AADHAAR_NUMBER"],
    "PAN": ["PAN", "PAN_NUMBER"],
    "BSN": ["BSN", "BSN_NUMBER"],
}

print("\nChecking which semantic groups have multiple labels in dataset:\n")

for group_name, patterns in SEMANTIC_PATTERNS.items():
    found_labels: list[tuple[str, int]] = [
        (label, all_labels[label])
        for label in patterns
        if label in all_labels
    ]
    if len(found_labels) > 1:
        total_count: int = sum(count for _, count in found_labels)
        print(f"{group_name} (total: {total_count:,}):")
        for label, count in sorted(found_labels, key=lambda x: -x[1]):
            print(f"    {label:40s} {count:,}")
        print()

# =============================================================================
# Save Label Inventory for Reference
# =============================================================================

# Store the label inventory for use in subsequent cells
LABEL_INVENTORY: dict[str, int] = dict(all_labels)

print("\n" + "=" * 80)
print("Analysis complete. LABEL_INVENTORY dictionary is now available.")
print("=" * 80)

Loading dataset from: Ari-S-123/better-english-pii-anonymizer

Dataset structure:
  Train split: 125,327 examples
  Test split: 31,361 examples

Extracting labels from train split...
Extracting labels from test split...

LABEL INVENTORY ANALYSIS

Total unique labels: 450
Total entity occurrences: 408,872

--- Top 50 Most Frequent Labels ---
    1. GIVENNAME                                 120,190 (29.40%)
    2. SURNAME                                    46,595 (11.40%)
    3. TIME                                       25,318 ( 6.19%)
    4. CITY                                       25,217 ( 6.17%)
    5. TELEPHONENUM                               23,468 ( 5.74%)
    6. DATE                                       22,275 ( 5.45%)
    7. EMAIL                                      18,319 ( 4.48%)
    8. STREET                                     18,147 ( 4.44%)
    9. TITLE                                      17,048 ( 4.17%)
   10. BUILDINGNUM                                15,607 ( 3.82

## Consolidated Label Mapping

Defines the canonical label schema and creates a comprehensive mapping from all variant labels (250+) to the
consolidated set (~35 labels).

Design Principles:

1. Use ai4privacy's canonical labels as the base (they have the most training data)
2. Map synthetic data labels to their ai4privacy equivalents
3. Consolidate semantically identical labels (e.g., PHONE_NUMBER -> PHONENUMBER)
4. Preserve label granularity where it matters (e.g., keep FIRSTNAME vs LASTNAME)
5. Ensure compatibility with benchmark models (piiranha, Isotonic models)

The consolidated schema targets approximately 35-40 distinct entity types, which is sufficient for high-quality PII
detection while avoiding fragmentation.


In [5]:
from typing import Final

# =============================================================================
# CANONICAL LABEL SET
# =============================================================================
# These are the final, consolidated labels that will be used in the dataset.
# Based on ai4privacy's 54-label schema, with some consolidation for clarity.

CANONICAL_LABELS: Final[list[str]] = [
    # Person Names
    "FIRSTNAME",        # Given name / first name
    "LASTNAME",         # Family name / surname
    "MIDDLENAME",       # Middle name
    "PREFIX",           # Title (Mr., Mrs., Dr., etc.)
    
    # Personal Attributes
    "AGE",              # Age (numeric)
    "GENDER",           # Gender identity
    "SEX",              # Biological sex
    "DOB",              # Date of birth
    "HEIGHT",           # Physical height
    "EYECOLOR",         # Eye color
    
    # Contact Information
    "EMAIL",            # Email address
    "PHONENUMBER",      # Phone number (any format)
    "USERNAME",         # Social media / online username
    
    # Location / Address
    "STREET",           # Street name
    "BUILDINGNUMBER",   # Building/house number
    "SECONDARYADDRESS", # Apartment, suite, unit
    "CITY",             # City name
    "STATE",            # State / province / region
    "COUNTY",           # County
    "ZIPCODE",          # Postal/ZIP code
    "NEARBYGPSCOORDINATE",  # GPS coordinates
    "ORDINALDIRECTION", # Cardinal direction (N, S, E, W)
    
    # Financial - Cards & Accounts
    "CREDITCARDNUMBER", # Credit/debit card number
    "CREDITCARDISSUER", # Card issuer (Visa, Mastercard, etc.)
    "CVV",              # Card security code
    "ACCOUNTNUMBER",    # Bank account number
    "ACCOUNTNAME",      # Account name
    "IBAN",             # International Bank Account Number
    "BIC",              # Bank Identifier Code / SWIFT
    
    # Financial - Currency
    "CURRENCY",         # Currency name (Dollar, Euro, etc.)
    "CURRENCYCODE",     # Currency code (USD, EUR, etc.)
    "CURRENCYSYMBOL",   # Currency symbol ($, €, etc.)
    "AMOUNT",           # Monetary amount
    
    # Government IDs
    "SSN",              # Social Security Number (US)
    "DRIVERLICENSENUM", # Driver's license number
    "PASSPORTNUM",      # Passport number
    "IDCARDNUM",        # Generic national ID card
    "TAXNUM",           # Tax identification number
    
    # Technology / Digital
    "IPV4",             # IPv4 address
    "IPV6",             # IPv6 address
    "MAC",              # MAC address
    "URL",              # Web URL
    "USERAGENT",        # Browser user agent string
    "PASSWORD",         # Password
    "PIN",              # PIN code
    
    # Device / Vehicle
    "PHONEIMEI",        # Phone IMEI number
    "VEHICLEVIN",       # Vehicle Identification Number
    "VEHICLEVRM",       # Vehicle registration mark
    
    # Cryptocurrency
    "BITCOINADDRESS",   # Bitcoin wallet address
    "LITECOINADDRESS",  # Litecoin wallet address
    "ETHEREUMADDRESS",  # Ethereum wallet address
    
    # Regional / Specialized IDs
    "AADHAAR",          # India Aadhaar number
    "UPI_ID",           # India UPI ID
    "NATIONAL_INSURANCE", # UK National Insurance Number
    
    # Employment
    "JOBTITLE",         # Job title
    "JOBTYPE",          # Job type
    "JOBAREA",          # Job area/department
    "COMPANYNAME",      # Company name
    
    # Temporal
    "DATE",             # Generic date
    "TIME",             # Time
    
    # Other
    "MASKEDNUMBER",     # Generic masked number
    "TITLE",            # Document/content title
]

# =============================================================================
# COMPREHENSIVE LABEL MAPPING
# =============================================================================
# Maps ALL variant labels to their canonical equivalents.
# Format: "VARIANT_LABEL": "CANONICAL_LABEL"
#
# Labels that map to themselves are canonical labels.
# Labels that map to None will be DROPPED (rare edge cases).

LABEL_MAPPING: Final[dict[str, str | None]] = {
    # =========================================================================
    # PERSON NAMES
    # =========================================================================
    # Given Names -> FIRSTNAME
    "FIRSTNAME": "FIRSTNAME",
    "FIRST_NAME": "FIRSTNAME",
    "GIVENNAME": "FIRSTNAME",
    "GIVEN_NAME": "FIRSTNAME",
    "PERSON_FIRST_NAME": "FIRSTNAME",
    "PERSON_GIVEN_NAME": "FIRSTNAME",
    
    # Family Names -> LASTNAME
    "LASTNAME": "LASTNAME",
    "LAST_NAME": "LASTNAME",
    "SURNAME": "LASTNAME",
    "FAMILY_NAME": "LASTNAME",
    "PERSON_LAST_NAME": "LASTNAME",
    
    # Middle Names -> MIDDLENAME
    "MIDDLENAME": "MIDDLENAME",
    "MIDDLE_INITIAL": "MIDDLENAME",
    
    # Prefixes/Titles -> PREFIX
    "PREFIX": "PREFIX",
    "TITLE": "TITLE",
    
    # Full/Generic Names -> FIRSTNAME (best approximation for NER)
    # Note: Full names should ideally be split, but we map to FIRSTNAME
    # as the model will learn to tag continuous name spans
    "NAME": "FIRSTNAME",
    "FULL_NAME": "FIRSTNAME",
    "PERSON_NAME": "FIRSTNAME",
    "PERSON_NAMES": "FIRSTNAME",
    "PERSONAL_NAME": "FIRSTNAME",
    "PERSONAL_NAMES": "FIRSTNAME",
    "PERSON_FULLNAME": "FIRSTNAME",
    "PERSON_FULL_NAME": "FIRSTNAME",
    "PERSON": "FIRSTNAME",
    "PERSON_ID": "IDCARDNUM",
    
    # =========================================================================
    # PERSONAL ATTRIBUTES
    # =========================================================================
    "AGE": "AGE",
    "GENDER": "GENDER",
    "SEX": "SEX",
    "DOB": "DOB",
    "DATE_OF_BIRTH": "DOB",
    "HEIGHT": "HEIGHT",
    "EYECOLOR": "EYECOLOR",
    
    # =========================================================================
    # CONTACT INFORMATION
    # =========================================================================
    # Email -> EMAIL
    "EMAIL": "EMAIL",
    "EMAIL_ADDRESS": "EMAIL",
    
    # Phone -> PHONENUMBER
    "PHONENUMBER": "PHONENUMBER",
    "PHONE": "PHONENUMBER",
    "PHONE_NUMBER": "PHONENUMBER",
    "PHONE_NUMBERS": "PHONENUMBER",
    "TELEPHONENUM": "PHONENUMBER",
    "UK_PHONE": "PHONENUMBER",
    "PHONE_EXT": "PHONENUMBER",
    
    # Usernames -> USERNAME
    "USERNAME": "USERNAME",
    "USERNAME_HANDLE": "USERNAME",
    "MODERN_USERNAME": "USERNAME",
    "DISCORD_HANDLE": "USERNAME",
    "DISCORD_TAG": "USERNAME",
    "VENMO_HANDLE": "USERNAME",
    "SOCIAL_MEDIA_HANDLE": "USERNAME",
    
    # =========================================================================
    # LOCATION / ADDRESS
    # =========================================================================
    # Street -> STREET
    "STREET": "STREET",
    "STREETADDRESS": "STREET",
    "STREET_ADDRESS": "STREET",
    
    # Building Number -> BUILDINGNUMBER
    "BUILDINGNUMBER": "BUILDINGNUMBER",
    "BUILDINGNUM": "BUILDINGNUMBER",
    
    # Secondary Address -> SECONDARYADDRESS
    "SECONDARYADDRESS": "SECONDARYADDRESS",
    
    # Full Address -> STREET (as primary component)
    "ADDRESS": "STREET",
    "POSTAL_ADDRESS": "STREET",
    "LOCATION_ADDRESS": "STREET",
    "US_STREET_ADDRESS": "STREET",
    "ADDRESS_RESIDENTIAL": "STREET",
    "LOCATION": "CITY",  # Location often refers to city/place
    
    # City -> CITY
    "CITY": "CITY",
    "HOTEL_NAMES": "CITY",  # Often contextually a place name
    
    # State/Province -> STATE
    "STATE": "STATE",
    "PROVINCE": "STATE",
    
    # County -> COUNTY
    "COUNTY": "COUNTY",
    
    # Postal/ZIP Code -> ZIPCODE
    "ZIPCODE": "ZIPCODE",
    "POSTAL_CODE": "ZIPCODE",
    "POSTAL_CODES": "ZIPCODE",
    "POSTCODE": "ZIPCODE",
    "POST_CODE": "ZIPCODE",
    "UK_POSTCODE": "ZIPCODE",
    "US_ZIP_CODE": "ZIPCODE",
    "PIN_CODE": "ZIPCODE",
    
    # GPS -> NEARBYGPSCOORDINATE
    "NEARBYGPSCOORDINATE": "NEARBYGPSCOORDINATE",
    
    # Direction -> ORDINALDIRECTION
    "ORDINALDIRECTION": "ORDINALDIRECTION",
    
    # Place Names (contextual) -> mapped to nearest equivalent
    "NAMES_OF_PLACES_OR_NOUNS": "CITY",
    "PLACE_OF_BIRTH": "CITY",
    
    # =========================================================================
    # FINANCIAL - CARDS & ACCOUNTS
    # =========================================================================
    # Credit Card Number -> CREDITCARDNUMBER
    "CREDITCARDNUMBER": "CREDITCARDNUMBER",
    "CREDIT_CARD": "CREDITCARDNUMBER",
    
    # Credit Card Issuer -> CREDITCARDISSUER
    "CREDITCARDISSUER": "CREDITCARDISSUER",
    "CARD_BRAND": "CREDITCARDISSUER",
    
    # CVV -> CVV
    "CVV": "CVV",
    "CREDITCARDCVV": "CVV",
    "CREDIT_CARD_CVV": "CVV",
    
    # Card Expiry -> DATE (since it's a date)
    "CARD_EXPIRY": "DATE",
    "CREDIT_CARD_EXPIRY": "DATE",
    "EXPIRATION_DATE": "DATE",
    "EXPIRY_DATE": "DATE",
    "EXP_DATE": "DATE",
    "DATE_OF_EXPIRATION": "DATE",
    "EFFECTIVE_DATE": "DATE",
    "ISSUE_DATE": "DATE",
    "PASSPORT_EXPIRY_DATE": "DATE",
    
    # Account Number -> ACCOUNTNUMBER
    "ACCOUNTNUMBER": "ACCOUNTNUMBER",
    "ACCOUNT_NUMBER": "ACCOUNTNUMBER",
    "BANK_ACCOUNT": "ACCOUNTNUMBER",
    "BANK_ACCOUNT_NUMBER": "ACCOUNTNUMBER",
    "BANK_ACCOUNT_NO": "ACCOUNTNUMBER",
    "BANK_ACCOUNT_IBAN": "IBAN",  # This is specifically IBAN
    "ACCOUNT": "ACCOUNTNUMBER",
    "ACCOUNT_ID": "ACCOUNTNUMBER",
    
    # Account Name -> ACCOUNTNAME
    "ACCOUNTNAME": "ACCOUNTNAME",
    
    # IBAN -> IBAN
    "IBAN": "IBAN",
    "IBAN_CODE": "IBAN",
    "BANK_IBAN": "IBAN",
    
    # BIC/SWIFT -> BIC
    "BIC": "BIC",
    "SWIFT_BIC": "BIC",
    "SWIFT_CODE": "BIC",
    
    # =========================================================================
    # FINANCIAL - CURRENCY
    # =========================================================================
    "CURRENCY": "CURRENCY",
    "CURRENCYNAME": "CURRENCY",
    "CURRENCYCODE": "CURRENCYCODE",
    "CURRENCYSYMBOL": "CURRENCYSYMBOL",
    "AMOUNT": "AMOUNT",
    "MONEY": "AMOUNT",
    "PRICE": "AMOUNT",
    
    # =========================================================================
    # GOVERNMENT IDs
    # =========================================================================
    # SSN -> SSN
    "SSN": "SSN",
    "SOCIALNUM": "SSN",
    "SOCIAL_SECURITY_NUMBER": "SSN",
    "US_SSN": "SSN",
    "SSN_LAST4": "SSN",
    "SSN_LAST_FOUR": "SSN",
    "US_SSN_LASTFOUR": "SSN",
    "NATIONAL_IDENTITY_SSN": "SSN",
    "NATIONAL_IDENTITY_SSN_AADHAR": "SSN",  # Consolidated for simplicity
    
    # Driver's License -> DRIVERLICENSENUM
    "DRIVERLICENSENUM": "DRIVERLICENSENUM",
    "DRIVERLICENSE": "DRIVERLICENSENUM",
    "DRIVER_LICENSE": "DRIVERLICENSENUM",
    "DRIVERS_LICENSE": "DRIVERLICENSENUM",
    "DRIVERS_LICENCE": "DRIVERLICENSENUM",
    "DRIVER_LICENCE": "DRIVERLICENSENUM",
    "DRIVERS_LICENSE_NUMBER": "DRIVERLICENSENUM",
    "UK_DRIVING_LICENCE": "DRIVERLICENSENUM",
    "UK_DRIVING_LICENCE_DATE": "DATE",  # This is a date, not a license number
    
    # Passport -> PASSPORTNUM
    "PASSPORTNUM": "PASSPORTNUM",
    "PASSPORT": "PASSPORTNUM",
    "PASSPORT_NUMBER": "PASSPORTNUM",
    "PASSPORT_NUMBERS": "PASSPORTNUM",
    "PASSPORT_ID": "PASSPORTNUM",
    "UK_PASSPORT": "PASSPORTNUM",
    "GERMAN_PASSPORT": "PASSPORTNUM",
    
    # National ID Card -> IDCARDNUM
    "IDCARDNUM": "IDCARDNUM",
    "ID_CARD": "IDCARDNUM",
    "ID_CARD_NUMBER": "IDCARDNUM",
    "ID_NUMBER": "IDCARDNUM",
    "ID_NUMBERS": "IDCARDNUM",
    "ID": "IDCARDNUM",
    "GOVT_ID": "IDCARDNUM",
    "NATIONAL_ID": "IDCARDNUM",
    "NATIONAL_IDENTITY": "IDCARDNUM",
    "NATIONALID": "IDCARDNUM",
    "NATIONAL_IDENTITY_PAN": "IDCARDNUM",
    "NATIONAL_ID_PAN": "IDCARDNUM",
    "OTHER_NATIONAL_IDENTITY": "IDCARDNUM",
    "DOCUMENT_ID": "IDCARDNUM",
    
    # Tax ID -> TAXNUM
    "TAXNUM": "TAXNUM",
    "TAX_ID": "TAXNUM",
    "TAX_IDENTIFICATION": "TAXNUM",
    "TFN": "TAXNUM",  # Australian Tax File Number
    "FISCAL_CODE": "TAXNUM",
    "IT_FISCAL_CODE": "TAXNUM",
    "SIREN": "TAXNUM",  # French business ID
    "SIRET": "TAXNUM",  # French business ID
    "VAT_ID": "TAXNUM",
    "VAT_NUMBER": "TAXNUM",
    
    # =========================================================================
    # TECHNOLOGY / DIGITAL
    # =========================================================================
    # IP Addresses
    "IPV4": "IPV4",
    "IP": "IPV4",  # Generic IP -> IPv4
    "IP_ADDRESS": "IPV4",
    "IPV6": "IPV6",
    
    # MAC Address -> MAC
    "MAC": "MAC",
    
    # URL -> URL
    "URL": "URL",
    
    # User Agent -> USERAGENT
    "USERAGENT": "USERAGENT",
    
    # Passwords/Secrets -> PASSWORD
    "PASSWORD": "PASSWORD",
    "API_KEY": "PASSWORD",
    "CLOUD_ID": "PASSWORD",
    
    # PIN -> PIN
    "PIN": "PIN",
    
    # 2FA/OTP Codes -> PIN (they're short numeric codes)
    "TFA_CODE": "PIN",
    "2FA_CODE": "PIN",
    "TWO_FACTOR_CODE": "PIN",
    "TWO_FA_CODE": "PIN",
    "TXN_2FA_CODE": "PIN",
    "OTP": "PIN",
    "OTP_CODE": "PIN",
    "TOTP": "PIN",
    "TOTP_CODE": "PIN",
    "VERIFICATION_OTP": "PIN",
    "AUTH_CODE": "PIN",
    "AUTH_TOKEN": "PASSWORD",  # Auth tokens are longer
    "INTERAC_CODE": "PIN",
    
    # =========================================================================
    # DEVICE / VEHICLE
    # =========================================================================
    # Phone IMEI -> PHONEIMEI
    "PHONEIMEI": "PHONEIMEI",
    
    # Vehicle IDs
    "VEHICLEVIN": "VEHICLEVIN",
    "VEHICLE_VIN": "VEHICLEVIN",
    "VEHICLE_ID": "VEHICLEVIN",
    "VEHICLE_IDENTIFICATION_NUMBER": "VEHICLEVIN",
    "VEHICLE_ID_NUMBER": "VEHICLEVIN",
    "VEHICLE_CHASSIS": "VEHICLEVIN",
    
    "VEHICLEVRM": "VEHICLEVRM",
    "VEHICLE_REG": "VEHICLEVRM",
    "VEHICLE_REGISTRATION": "VEHICLEVRM",
    
    # =========================================================================
    # CRYPTOCURRENCY
    # =========================================================================
    "BITCOINADDRESS": "BITCOINADDRESS",
    "BITCOIN_ADDRESS": "BITCOINADDRESS",
    "BITCOIN_WALLET": "BITCOINADDRESS",
    
    "LITECOINADDRESS": "LITECOINADDRESS",
    
    "ETHEREUMADDRESS": "ETHEREUMADDRESS",
    "ETHEREUM_WALLET": "ETHEREUMADDRESS",
    
    # Generic Crypto -> BITCOINADDRESS (most common)
    "CRYPTO_WALLET": "BITCOINADDRESS",
    "CRYPTO_WALLET_ADDRESS": "BITCOINADDRESS",
    "CRYPTO_WALLET_ETHEREUM": "ETHEREUMADDRESS",
    "CRYPTO_TXID": "BITCOINADDRESS",  # Transaction IDs
    "CRYPTO_TXN_HASH": "BITCOINADDRESS",
    "CRYPTO_TXN_ID": "BITCOINADDRESS",
    "TX_HASH": "BITCOINADDRESS",
    
    # =========================================================================
    # REGIONAL / SPECIALIZED IDs
    # =========================================================================
    # India Aadhaar
    "AADHAAR": "AADHAAR",
    "AADHAAR_NUMBER": "AADHAAR",
    
    # India PAN -> IDCARDNUM (it's a national ID)
    "PAN": "IDCARDNUM",
    "PAN_NUMBER": "IDCARDNUM",
    
    # India UPI
    "UPI_ID": "UPI_ID",
    "UPI": "UPI_ID",
    "BANK_UPI_ID": "UPI_ID",
    "UPI_TRANSACTION_ID": "UPI_ID",
    "DIGITAL_PAYMENT_ID": "UPI_ID",
    "PAYMENT_ID": "UPI_ID",
    
    # UK National Insurance
    "NATIONAL_INSURANCE": "NATIONAL_INSURANCE",
    "NATIONAL_INSURANCE_NUMBER": "NATIONAL_INSURANCE",
    "UK_NATIONAL_INSURANCE_NUMBER": "NATIONAL_INSURANCE",
    "UK_NI": "NATIONAL_INSURANCE",
    "UK_NI_NUMBER": "NATIONAL_INSURANCE",
    
    # UK NHS Number -> IDCARDNUM
    "UK_NHS_NUMBER": "IDCARDNUM",
    
    # Netherlands BSN -> IDCARDNUM
    "BSN": "IDCARDNUM",
    "BSN_NUMBER": "IDCARDNUM",
    
    # Netherlands DigiD -> IDCARDNUM
    "DIGID": "IDCARDNUM",
    
    # Spain NIE -> IDCARDNUM
    "NIE": "IDCARDNUM",
    
    # =========================================================================
    # EMPLOYMENT
    # =========================================================================
    "JOBTITLE": "JOBTITLE",
    "JOBTYPE": "JOBTYPE",
    "JOBAREA": "JOBAREA",
    "COMPANYNAME": "COMPANYNAME",
    "ORG": "COMPANYNAME",
    "ORGANIZATION": "COMPANYNAME",
    "ORGANIZATIONS": "COMPANYNAME",
    "ORGANIZATION_ID": "COMPANYNAME",
    "ORGANIZATION_VAT_ID": "TAXNUM",
    
    # =========================================================================
    # TEMPORAL
    # =========================================================================
    "DATE": "DATE",
    "DATES": "DATE",
    "TIME": "TIME",
    
    # =========================================================================
    # INSURANCE
    # =========================================================================
    "INSURANCE": "IDCARDNUM",
    "INSURANCE_NUMBER": "IDCARDNUM",
    "INSURANCE_POLICY": "IDCARDNUM",
    "INSURANCE_POLICY_NUMBER": "IDCARDNUM",
    "MEDICARE_NUMBER": "IDCARDNUM",
    "HEALTH_CARD_NUMBER": "IDCARDNUM",
    "CLAIM_NUMBER": "IDCARDNUM",
    "POLICY_NUMBER": "IDCARDNUM",
    
    # =========================================================================
    # ORDER / TRACKING / MISC IDs
    # =========================================================================
    "MASKEDNUMBER": "MASKEDNUMBER",
    "ORDER": "MASKEDNUMBER",
    "ORDER_ID": "MASKEDNUMBER",
    "ORDER_NUMBER": "MASKEDNUMBER",
    "BOOKING_ID": "MASKEDNUMBER",
    "CONFIRMATION_NUMBER": "MASKEDNUMBER",
    "CONTRACT_NUMBER": "MASKEDNUMBER",
    "CUSTOMERID": "MASKEDNUMBER",
    "CUSTOMER_ID": "MASKEDNUMBER",
    "EMPLOYEE_ID": "MASKEDNUMBER",
    "MEMBER_ID": "MASKEDNUMBER",
    "PRODUCT_ID": "MASKEDNUMBER",
    "PRODUCT_SERIAL": "MASKEDNUMBER",
    "REFERENCE_NUMBER": "MASKEDNUMBER",
    "REFERRAL_CODE": "MASKEDNUMBER",
    "REGISTRATION_NUMBER": "MASKEDNUMBER",
    "SIN": "SSN",  # Canadian Social Insurance Number
    "STUDENT_ID": "MASKEDNUMBER",
    "STUD_ID": "MASKEDNUMBER",
    "TRACKING_NUMBER": "MASKEDNUMBER",
    "TRACK_NUM": "MASKEDNUMBER",
    "TRANSACTION_ID": "MASKEDNUMBER",
    "TXN_ID": "MASKEDNUMBER",
    "DISCOUNT_CODE": "MASKEDNUMBER",
    "MECHANIC_CODE": "MASKEDNUMBER",
    "AIRPORT_CODE": "MASKEDNUMBER",
    "IFSC": "BIC",  # Indian bank branch code (similar to SWIFT/BIC)
    "EXTENSION": "PHONENUMBER",  # Phone extension
}

# =============================================================================
# VALIDATION
# =============================================================================

# Ensure all canonical labels are in the mapping
for label in CANONICAL_LABELS:
    if label not in LABEL_MAPPING:
        print(f"WARNING: Canonical label '{label}' not in LABEL_MAPPING!")
        LABEL_MAPPING[label] = label

# Count how many source labels map to each canonical label
canonical_sources: dict[str, list[str]] = {}
for source, target in LABEL_MAPPING.items():
    if target is not None:
        if target not in canonical_sources:
            canonical_sources[target] = []
        canonical_sources[target].append(source)

print("=" * 80)
print("CONSOLIDATED LABEL MAPPING SUMMARY")
print("=" * 80)
print(f"\nTotal source labels mapped: {len(LABEL_MAPPING)}")
print(f"Total canonical labels: {len(set(v for v in LABEL_MAPPING.values() if v))}")

print("\n--- Canonical labels with multiple sources ---")
for canonical, sources in sorted(canonical_sources.items(), key=lambda x: -len(x[1])):
    if len(sources) > 1:
        print(f"\n{canonical} ({len(sources)} sources):")
        for src in sorted(sources):
            if src != canonical:
                print(f"    <- {src}")

print("\n" + "=" * 80)
print("LABEL_MAPPING dictionary is now available for use.")
print("=" * 80)

CONSOLIDATED LABEL MAPPING SUMMARY

Total source labels mapped: 279
Total canonical labels: 62

--- Canonical labels with multiple sources ---

IDCARDNUM (30 sources):
    <- BSN
    <- BSN_NUMBER
    <- CLAIM_NUMBER
    <- DIGID
    <- DOCUMENT_ID
    <- GOVT_ID
    <- HEALTH_CARD_NUMBER
    <- ID
    <- ID_CARD
    <- ID_CARD_NUMBER
    <- ID_NUMBER
    <- ID_NUMBERS
    <- INSURANCE
    <- INSURANCE_NUMBER
    <- INSURANCE_POLICY
    <- INSURANCE_POLICY_NUMBER
    <- MEDICARE_NUMBER
    <- NATIONALID
    <- NATIONAL_ID
    <- NATIONAL_IDENTITY
    <- NATIONAL_IDENTITY_PAN
    <- NATIONAL_ID_PAN
    <- NIE
    <- OTHER_NATIONAL_IDENTITY
    <- PAN
    <- PAN_NUMBER
    <- PERSON_ID
    <- POLICY_NUMBER
    <- UK_NHS_NUMBER

MASKEDNUMBER (25 sources):
    <- AIRPORT_CODE
    <- BOOKING_ID
    <- CONFIRMATION_NUMBER
    <- CONTRACT_NUMBER
    <- CUSTOMERID
    <- CUSTOMER_ID
    <- DISCOUNT_CODE
    <- EMPLOYEE_ID
    <- MECHANIC_CODE
    <- MEMBER_ID
    <- ORDER
    <- ORDER_ID
    <

## Transform the Dataset with Consolidated Labels

Applies the LABEL_MAPPING to transform the dataset from 250+ fragmented labels to the consolidated ~35 canonical labels.

This transformation:

1. Maps all entity labels to their canonical equivalents
2. Handles unknown labels gracefully (logs them and optionally drops)
3. Validates the transformation results
4. Saves the consolidated dataset locally and pushes to HuggingFace Hub


In [6]:
import json
from collections import Counter
from pathlib import Path
from typing import Any

from datasets import Dataset, DatasetDict, load_dataset

# =============================================================================
# Configuration
# =============================================================================

INPUT_DATASET_ID: str = "Ari-S-123/better-english-pii-anonymizer"
OUTPUT_DATASET_ID: str = "Ari-S-123/pii-detection-english-consolidated"  # New name
OUTPUT_PATH: Path = Path("./data/consolidated_dataset")

# How to handle labels not in LABEL_MAPPING
# Options: "drop" (remove the entity), "keep" (keep original label), "error" (raise)
UNKNOWN_LABEL_STRATEGY: str = "keep"

# =============================================================================
# Load Original Dataset
# =============================================================================

print("Loading dataset from: {}".format(INPUT_DATASET_ID))
original_dataset: DatasetDict = load_dataset(INPUT_DATASET_ID)

print("\nOriginal dataset structure:")
print("  Train split: {:,} examples".format(len(original_dataset["train"])))
print("  Test split: {:,} examples".format(len(original_dataset["test"])))

# =============================================================================
# Transformation Function
# =============================================================================

# Track unknown labels encountered during transformation
unknown_labels_encountered: Counter = Counter()


def transform_example(example: dict[str, Any]) -> dict[str, Any]:
    """
    Transform a single example by mapping all entity labels to canonical labels.

    This function processes the 'privacy_mask' field and remaps each entity's
    label according to LABEL_MAPPING. Unknown labels are handled according to
    UNKNOWN_LABEL_STRATEGY.

    Args:
        example: A single dataset example containing 'privacy_mask' and other fields.

    Returns:
        The transformed example with consolidated labels.

    Side Effects:
        Updates unknown_labels_encountered counter for tracking.
    """
    privacy_mask: list[dict[str, Any]] = example.get("privacy_mask", [])

    if not privacy_mask:
        return example

    # Transform each entity in the privacy mask
    transformed_mask: list[dict[str, Any]] = []

    for entity in privacy_mask:
        original_label: str = entity.get("label", "UNKNOWN")

        # Look up the canonical label
        if original_label in LABEL_MAPPING:
            canonical_label: str | None = LABEL_MAPPING[original_label]

            if canonical_label is None:
                # Label is explicitly marked for dropping
                continue
            else:
                # Create transformed entity
                transformed_entity: dict[str, Any] = entity.copy()
                transformed_entity["label"] = canonical_label
                transformed_mask.append(transformed_entity)
        else:
            # Unknown label not in mapping
            unknown_labels_encountered[original_label] += 1

            if UNKNOWN_LABEL_STRATEGY == "drop":
                continue
            elif UNKNOWN_LABEL_STRATEGY == "keep":
                transformed_mask.append(entity)
            elif UNKNOWN_LABEL_STRATEGY == "error":
                raise ValueError("Unknown label encountered: {}".format(original_label))

    # Return the transformed example
    transformed_example: dict[str, Any] = example.copy()
    transformed_example["privacy_mask"] = transformed_mask

    return transformed_example


# =============================================================================
# Apply Transformation
# =============================================================================

print("\nTransforming train split...")
transformed_train: Dataset = original_dataset["train"].map(
    transform_example,
    desc="Consolidating train labels",
)

print("Transforming test split...")
transformed_test: Dataset = original_dataset["test"].map(
    transform_example,
    desc="Consolidating test labels",
)

# =============================================================================
# Report Unknown Labels
# =============================================================================

if unknown_labels_encountered:
    print("\n" + "=" * 80)
    print("WARNING: Unknown labels encountered (not in LABEL_MAPPING)")
    print("=" * 80)
    print("Strategy used: {}".format(UNKNOWN_LABEL_STRATEGY))
    print("\nUnknown labels ({} unique):".format(len(unknown_labels_encountered)))
    for label, count in unknown_labels_encountered.most_common():
        print("  {:<45} {:,}".format(label, count))
    
    print("\nConsider adding these to LABEL_MAPPING in Cell 2.")

# =============================================================================
# Validate Transformation Results
# =============================================================================

print("\n" + "=" * 80)
print("TRANSFORMATION VALIDATION")
print("=" * 80)


def count_labels_in_dataset(dataset: Dataset) -> Counter:
    """
    Count all unique labels in a transformed dataset.

    Args:
        dataset: A HuggingFace Dataset with transformed 'privacy_mask' field.

    Returns:
        Counter mapping label names to occurrence counts.
    """
    label_counter: Counter = Counter()
    for example in dataset:
        for entity in example.get("privacy_mask", []):
            label_counter[entity.get("label", "UNKNOWN")] += 1
    return label_counter


train_labels: Counter = count_labels_in_dataset(transformed_train)
test_labels: Counter = count_labels_in_dataset(transformed_test)
all_labels: Counter = train_labels + test_labels

total_entities: int = sum(all_labels.values())

print("\nAfter consolidation:")
print("  Unique labels: {}".format(len(all_labels)))
print("  Total entities: {:,}".format(total_entities))

print("\n--- Label Distribution (Top 30) ---")
for label, count in all_labels.most_common(30):
    pct: float = (count / total_entities) * 100
    print("  {:<30} {:>8,} ({:>5.2f}%)".format(label, count, pct))

# =============================================================================
# Create Consolidated DatasetDict
# =============================================================================

consolidated_dataset: DatasetDict = DatasetDict({
    "train": transformed_train,
    "test": transformed_test,
})

print("\nConsolidated dataset created:")
print("  Train: {:,} examples".format(len(consolidated_dataset["train"])))
print("  Test: {:,} examples".format(len(consolidated_dataset["test"])))

# =============================================================================
# Save Locally
# =============================================================================

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print("\nSaving to: {}".format(OUTPUT_PATH))

# Save as Parquet (recommended format)
consolidated_dataset["train"].to_parquet(OUTPUT_PATH / "train.parquet")
consolidated_dataset["test"].to_parquet(OUTPUT_PATH / "test.parquet")

# Save label mapping for reference
with open(OUTPUT_PATH / "label_mapping.json", "w") as f:
    json.dump(LABEL_MAPPING, f, indent=2)

# Save canonical labels list
canonical_labels_list: list[str] = sorted(set(v for v in LABEL_MAPPING.values() if v))
with open(OUTPUT_PATH / "canonical_labels.json", "w") as f:
    json.dump(canonical_labels_list, f, indent=2)

print("Local save complete.")

# =============================================================================
# Push to HuggingFace Hub (Optional)
# =============================================================================

PUSH_TO_HUB: bool = True  # Set to True to push

if PUSH_TO_HUB:
    print("\nPushing to HuggingFace Hub: {}".format(OUTPUT_DATASET_ID))
    
    consolidated_dataset.push_to_hub(
        OUTPUT_DATASET_ID,
        private=False,
    )
    
    print("Upload complete!")
    print("View at: https://huggingface.co/datasets/{}".format(OUTPUT_DATASET_ID))

print("\n" + "=" * 80)
print("TRANSFORMATION COMPLETE")
print("=" * 80)

Loading dataset from: Ari-S-123/better-english-pii-anonymizer

Original dataset structure:
  Train split: 125,327 examples
  Test split: 31,361 examples

Transforming train split...


Consolidating train labels:   0%|          | 0/125327 [00:00<?, ? examples/s]

Transforming test split...


Consolidating test labels:   0%|          | 0/31361 [00:00<?, ? examples/s]


Strategy used: keep

Unknown labels (208 unique):
  NAMES                                         15
  TAX_ID_PAN                                    9
  US_SOCIAL_SECURITY_NUMBER                     8
  SORT_CODE                                     8
  CRYPTO_TX_HASH                                7
  BANK_NAME                                     7
  UK_PHONE_NUMBER                               7
  IFSC_CODE                                     7
  CVC                                           6
  BANK_IFSC                                     6
  PRODUCT_CODE                                  6
  ETHEREUM_ADDRESS                              5
  INVOICE_NUMBER                                5
  ABN                                           5
  UK_NIN                                        5
  ORG_ID                                        5
  VERIFICATION_CODE                             5
  SSN_FR                                        4
  NI_NUMBER                                     

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Upload complete!
View at: https://huggingface.co/datasets/Ari-S-123/pii-detection-english-consolidated

TRANSFORMATION COMPLETE
